# Lesson 1 — Python basics for bioinformatics
**Date:** 30 September 2026  
**Goal:** go from zero to a working `reverse_complement()` function

| Topic | Status |
|---|---|
| Setup (Miniconda, Jupyter, ChimeraX, Jalview) | ✅ |
| Variables & strings | ✅ |
| Counting bases & GC content | ✅ |
| Slicing | ✅ |
| `for` loops | ✅ |
| `if` / `elif` | ✅ |
| Dictionaries | ✅ |
| Functions | ✅ |

> **How to use this notebook:** read the notes, then run each code cell with **Shift + Enter**.  
> Before running a cell, **predict the output**. That's the fastest way to learn.

---
## 0. Setup recap

### Starting work every time
Open **Anaconda Prompt** (Start menu), then:
```
conda activate thesis
jupyter lab
```

### What each tool is for
| Tool | What it does | Used for in the thesis |
|---|---|---|
| **Jupyter** | Write & run Python in small cells, results appear underneath | All analyses: sequences, mutation scores, kinetics plots |
| **Jalview** | View & colour sequence alignments | Finding conserved (often catalytic) residues |
| **ChimeraX** | 3D protein structure viewer | Seeing the active site, placing mutations, figures |

### UK keyboard — keys you need
| Symbol | Key |
|---|---|
| `"` | Shift + 2 |
| `'` | key left of Enter |
| `#` | key left of Enter (own key) |
| `[ ]` | right of P |
| `{ }` | Shift + `[ ]` |

---
## 1. How Jupyter works

- A notebook is made of **cells**. **Shift + Enter** runs a cell.
- Jupyter only knows what you have **actually run**, not what is written on screen.
  - An **orange dot** next to a cell = edited but not re-run.
  - After closing Jupyter, variables are forgotten → re-run the cells from the top.
- Only the **last line** of a cell is shown automatically. For more, use `print()`.
- Save with **Ctrl + S**.

> 🐞 **Today's bug:** I changed the DNA sequence in the first cell but didn't re-run it,  
> so all the counts were for the *old* sequence.

---
## 2. Variables & strings

A **variable** stores a value under a name. A **string** is text in quotes, so a DNA sequence is just a string.

**Naming rules**
- Only letters, numbers and `_` → `gc_percent` ✅, `GC%` ❌ (`%` is an operator)
- Python is **case-sensitive**: `dna` and `DNA` are two different variables
- Convention: lowercase names (`dna`, not `DNA`)
- `"ATGC"` and `'ATGC'` are the same, both quote types work

In [ ]:
dna = "AAAAAAAAACCCGGGTTTTTTTTTT"
dna

In [ ]:
len(dna)      # length of the sequence

In [ ]:
type(dna)     # str = string

---
## 3. Counting bases & GC content

`.count("X")` counts how many times X appears in a string.

In [ ]:
print("A:", dna.count("A"))
print("T:", dna.count("T"))
print("G:", dna.count("G"))
print("C:", dna.count("C"))

### GC content (%)

**GC % = (G + C) / length × 100**

Three rules learned the hard way:
1. `( )` is for grouping maths. `[ ]` creates a **list**, not grouping.
2. Python does `*` and `/` **before** `+`, so `100 * G + C` multiplies only G.
3. No `%` in variable names.

| Attempt | Problem |
|---|---|
| `GC% = [100 * G + C] / len(dna)` | `%` in name, `[ ]` instead of `( )`, order of operations |
| `GC = (100 * G + C) / len(dna)` | still only G × 100 → 12.12 ❌ |
| `GC = (100 * (G + C)) / len(dna)` | ✅ 24.0 |

**Clean version:** one step per line.

In [ ]:
g = dna.count("G")
c = dna.count("C")
gc = (g + c) / len(dna) * 100
gc        # expected: 24.0  (6 / 25 × 100)

---
## 4. Slicing

```
seq[start : stop : step]
```
- **start**: where to begin (included)
- **stop**: where to end (**not** included)
- **step**: jump size (1 = forward, −1 = backward)
- Empty slot = default (beginning / end / 1)

Positions can be counted from the front or the back:
```
letter:   A   C   G   T
front:    0   1   2   3
back:    -4  -3  -2  -1
```

| Code | Meaning | Result for `"ACGT"` |
|---|---|---|
| `s[:-1]` | stop before the last letter | `ACG` |
| `s[::-1]` | step backwards → **reverse** | `TGCA` |
| `s[-1:]` | start at the last letter | `T` |
| `s[1:3]` | positions 1 and 2 | `CG` |
| `s[::2]` | every 2nd letter | `AG` |

In [ ]:
s = "ACGT"
print(s[:-1])
print(s[::-1])
print(s[-1:])
print(s[1:3])
print(s[::2])

---
## 5. `for` loops

A loop repeats something for every **item** of a thing.
```python
for base in dna:
    print(base)
```
- `base` is just a name you choose, it could be anything.
- What counts as an "item" depends on what comes after `in`:
  - **string** → one letter at a time
  - **list** → one element at a time
- The `:` opens the loop, the **4-space indent** marks what's inside it.
- **Indentation decides when a line runs.** Not indented = runs once, after the loop.

In [ ]:
for base in "ACGT":
    print(base)
print("done")      # not indented → runs once

In [ ]:
for x in ["ATG", "TAA", "GGC"]:     # a list → whole elements
    print(x)

---
## 6. `if` / `elif`

```python
if base == "A":
    ...
elif base == "T":
    ...
```
- `==` asks "is it equal?" and `=` stores a value. Don't mix them up.
- `elif` = "else, if…" → it **needs its own condition**.
  - `elif rc = rc + "A"` ❌ (an action where the question should be)
  - `elif base == "T":` ✅
- Code inside an `if` inside a loop gets **8 spaces**.

---
## 7. Biology: the reverse complement

DNA strands are **antiparallel**, and sequences are always written **5′ → 3′**.
```
5′  A T G C G T A A C  3′   ← my strand
3′  T A C G C A T T G  5′   ← complementary strand
```
To write the other strand correctly (5′→3′) you need **both** steps:

| Step | Result |
|---|---|
| `dna` (5′→3′) | `ATGCGTAAC` |
| reverse → 3′→5′, still my strand | `CAATGCGTA` |
| complement → other strand, 5′→3′ | `GTTACGCAT` |

**Why it matters:** designing **reverse primers** (site-directed mutagenesis in the summer!) and reading genes on the opposite strand.

**Palindromes:** a sequence equal to its own reverse complement, like the EcoRI site `GAATTC`. Many restriction enzyme sites are palindromic.

### Version 1: `if` / `elif`

In [ ]:
dna = "ATGCGTAAC"

rc = ""
for base in dna[::-1]:
    if base == "A":
        rc = rc + "T"
    elif base == "T":
        rc = rc + "A"
    elif base == "G":
        rc = rc + "C"
    elif base == "C":
        rc = rc + "G"
print(rc)          # 0 spaces → prints once, after the loop.  Expected: GTTACGCAT

> 🐞 **Today's bug:** `print(rc)` was indented inside the `elif base == "C":` branch,  
> so it only printed when the base was C → output `G` and `GTTACG`.  
> The logic was right, the print was in the wrong place.

---
## 8. Dictionaries

A dictionary stores **key → value** pairs, like looking a word up in a real dictionary.
```python
pairs = {"A": "T", "T": "A", "G": "C", "C": "G"}
pairs["A"]   # → 'T'
```
- `{ }` creates it, `key: value`, pairs separated by commas
- Look up with `[ ]`
- A key that doesn't exist → **`KeyError`** (for example `pairs["X"]`, a lowercase `"a"`, or `"N"`)

**Quotes matter:**
| Code | Means |
|---|---|
| `pairs["base"]` | look up the literal word *base* → `KeyError` ❌ |
| `pairs[base]` | look up whatever letter the variable holds ✅ |

In [ ]:
pairs = {"A": "T", "T": "A", "G": "C", "C": "G"}
pairs["C"]

### Version 2: dictionary (one line instead of eight)

In [ ]:
rc = ""
for base in dna[::-1]:
    rc = rc + pairs[base]      # add the partner of the current letter
print(rc)                      # Expected: GTTACGCAT

| Attempt | Problem |
|---|---|
| `rc = pairs["base"]` | quotes → literal word; `=` overwrites instead of adding |
| `rc = pairs["A", "T", "C", "G"]` | no need to list keys, the loop already gives one letter at a time |
| `rc = rc + pairs[base]` | ✅ |

---
## 9. Functions

A **function** = a named, reusable block of code. Write once, use on any input.
```python
def name(parameter):
    ...
    return result
```
- `def` defines it. Running the `def` cell shows **nothing**, it only teaches Python the recipe.
- The **parameter** (`seq`) is a placeholder that only exists **inside** the function.
- You **call** it with an input: `reverse_complement(dna)`, and inside, `seq` = whatever you pass.
- `return` ends the function **immediately** and hands the value back.

### `return` vs `print`
| | `print` | `return` |
|---|---|---|
| Shows on screen | ✅ | only as the last line of a cell |
| Can be stored in a variable | ❌ (gives `None`) | ✅ |
| Stops the function | ❌ | ✅ |

In [ ]:
def gc_content(seq):
    g = seq.count("G")
    c = seq.count("C")
    return (g + c) / len(seq) * 100

print(gc_content("ATGCGTAAC"))
print(gc_content("GGGGCCCC"))

### The final function

It handles lowercase letters (`.upper()`) and unknown bases (`"N": "N"`).

In [ ]:
def reverse_complement(seq):
    seq = seq.upper()
    pairs = {"A": "T", "T": "A", "G": "C", "C": "G", "N": "N"}
    rc = ""
    for base in seq[::-1]:
        rc = rc + pairs[base]
    return rc            # 4 spaces: inside the function, after the loop

In [ ]:
print(reverse_complement("atgcgtaac"))   # expected: GTTACGCAT
print(reverse_complement("ATGNNCGT"))    # expected: ACGNNCAT
print(reverse_complement(dna))           # any variable works

> 🐞 **Today's bugs in this function:**
> 1. `for` and `return` not indented → `SyntaxError: 'return' outside function`
> 2. `"G": "T"` typo → **no error, just wrong answers** (the dangerous kind!)
> 3. adding to `seq` instead of a new `rc`
> 4. `return rc` inside the loop → function stopped after 1 letter

### Asking the user for input
`input()` shows a message and waits for typing. It's nice for small tools, but real sequences come from files and databases.
```python
user_seq = input("Enter sequence: ")
print(reverse_complement(user_seq))
```

---
## 10. Bug log: lessons from today's mistakes

| Bug | Lesson |
|---|---|
| Edited a cell but didn't re-run it | Jupyter only knows what you ran |
| `GC%` as a name | Names: letters, numbers, `_` only |
| `[ ]` for grouping | `( )` for maths, `[ ]` makes lists |
| `100 * G + C` | `*` happens before `+` |
| `dna` vs `DNA` | Python is case-sensitive |
| `print` inside the `if` | Indentation decides **when** code runs |
| `elif` without a condition | `elif` needs a question |
| `pairs["base"]` | Quotes = text, no quotes = variable |
| `rc = ...` instead of `rc = rc + ...` | Build strings by adding |
| `return` inside the loop | `return` stops everything |
| `"G": "T"` | Always test on a case you know the answer to |

---
## 11. Practice before next session

Try these on your own, without looking at the code above:

1. Write `complement(seq)`: the complement **without** reversing.  
   `complement("ATGC")` → `TACG`
2. Write `base_counts(seq)` that **returns a dictionary** of counts.  
   `base_counts("AATGC")` → `{'A': 2, 'T': 1, 'G': 1, 'C': 1}`
3. Write `is_palindrome(seq)` that returns `True` if a sequence equals its own reverse complement.  
   Test: `"GAATTC"` → `True`, `"ATGC"` → `True`, `"ATGCGTAAC"` → `False`

In [ ]:
# your code here

---
## 12. Next session: Biopython

- Biopython's built-in `reverse_complement()` (now I know what it does inside)
- Reading FASTA files
- Downloading a real **β-xylosidase** sequence from UniProt
- Length, molecular weight and composition of a real protein